# Aplicação base — agente de biblioteca

Este notebook contém uma aplicação de IA funcional que será utilizada no Hands-on do Capítulo 6. Execute as células em ordem e confirme que o agente responde antes de iniciar a migração para um projeto Python local.

O desafio do Hands-on não é reconstruir a lógica de IA. O objetivo será transformar esta aplicação em uma API com FastAPI, organizar o código em arquivos `.py` e versionar o projeto no GitHub.


## 1. Configuração no Google Colab

Adicione uma chave chamada `GOOGLE_API_KEY` aos Secrets do Google Colab antes de executar as próximas células.


In [ ]:
%pip install -q -U langchain langgraph langchain-google-genai


In [ ]:
import os

from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

NOME_MODELO = "google_genai:gemini-3.5-flash-lite"
model = init_chat_model(NOME_MODELO)

print("Modelo configurado:", NOME_MODELO)


## 2. Dados e funções da biblioteca

Os dicionários abaixo representam os dados utilizados pela aplicação. As funções consultam livros e empréstimos e permitem renovar um empréstimo ativo.


In [ ]:
livros = {
    "LIV-101": {
        "codigo": "LIV-101",
        "titulo": "Python Fluente",
        "paginas": 640
    },
    "LIV-202": {
        "codigo": "LIV-202",
        "titulo": "Clean Code",
        "paginas": 464
    }
}

emprestimos = {
    "EMP-500": {
        "emprestimo": "EMP-500",
        "status": "ativo",
        "codigo_livro": "LIV-101"
    }
}


In [ ]:
def consultar_livro(codigo: str) -> dict:
    '''Consulta título e quantidade de páginas de um livro pelo código.'''
    codigo = codigo.strip().upper()

    if not codigo.startswith("LIV-"):
        return {"erro": "código de livro inválido"}

    livro = livros.get(codigo)

    if livro is None:
        return {"erro": "livro não encontrado"}

    return livro.copy()


def consultar_emprestimo(emprestimo: str) -> dict:
    '''Consulta status e código do livro associado a um empréstimo.'''
    emprestimo = emprestimo.strip().upper()

    if not emprestimo.startswith("EMP-"):
        return {"erro": "identificador de empréstimo inválido"}

    registro = emprestimos.get(emprestimo)

    if registro is None:
        return {"erro": "empréstimo não encontrado"}

    return registro.copy()


def renovar_emprestimo(emprestimo: str) -> dict:
    '''Renova um empréstimo ativo e devolve seu novo estado.'''
    emprestimo = emprestimo.strip().upper()

    if not emprestimo.startswith("EMP-"):
        return {"erro": "identificador de empréstimo inválido"}

    registro = emprestimos.get(emprestimo)

    if registro is None:
        return {"erro": "empréstimo não encontrado"}

    if registro["status"] != "ativo":
        return {
            "erro": "empréstimo não pode ser renovado",
            "status_atual": registro["status"]
        }

    registro["status"] = "renovado"
    return registro.copy()


## 3. Construindo o agente

As funções Python são disponibilizadas como tools. O agente também utiliza memória de curto prazo para preservar o estado de cada conversa.


In [ ]:
consultar_livro_tool = tool(consultar_livro)
consultar_emprestimo_tool = tool(consultar_emprestimo)
renovar_emprestimo_tool = tool(renovar_emprestimo)

tools_biblioteca = [
    consultar_livro_tool,
    consultar_emprestimo_tool,
    renovar_emprestimo_tool,
]


In [ ]:
INSTRUCOES_BIBLIOTECA = """
Você atende usuários de uma biblioteca.

- Use as tools para consultar livros e empréstimos.
- Não invente livros, empréstimos, códigos ou resultados.
- Se faltar um identificador necessário, peça esse dado.
- Para descobrir informações do livro associado a um empréstimo,
  consulte primeiro o empréstimo e use o codigo_livro retornado.
- Use renovar_emprestimo somente quando o usuário pedir explicitamente
  a renovação.
- Responda de forma objetiva depois de obter os dados necessários.
"""

memoria = InMemorySaver()

agente_biblioteca = create_agent(
    model=model,
    tools=tools_biblioteca,
    system_prompt=INSTRUCOES_BIBLIOTECA,
    checkpointer=memoria,
)


## 4. Validando a aplicação

As duas chamadas abaixo utilizam o mesmo `thread_id`. A segunda mensagem depende do contexto criado na primeira. A redação da resposta pode variar entre execuções.


In [ ]:
config = {
    "configurable": {
        "thread_id": "bib-001"
    }
}

turno_1 = agente_biblioteca.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Tenho o empréstimo EMP-500. "
                    "A qual livro ele está associado?"
                ),
            }
        ]
    },
    config=config,
)

print(turno_1["messages"][-1].text)

turno_2 = agente_biblioteca.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Quantas páginas tem esse livro?",
            }
        ]
    },
    config=config,
)

print(turno_2["messages"][-1].text)
